In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


In [2]:
!python - <<'PY'
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    p = torch.cuda.get_device_properties(0)
    print("VRAM GiB:", round(p.total_memory / 1024**3, 3))
PY

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
Torch CUDA: 12.8
GPU: Tesla T4
VRAM GiB: 14.562


NameError: name 'PY' is not defined

In [3]:
!python -m pip install -q \
    "transformers==5.0.0" \
    "accelerate==1.13.0" \
    "bitsandbytes==0.50.2" \
    "peft==0.19.1" \
    "trl==1.13.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 37.2 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.7 MB/s eta 0:00:00


In [4]:
!python - <<'PY'
import torch
import transformers
import accelerate
import bitsandbytes
import peft
import trl

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
PY

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
Torch: 2.10.0+cu128
Torch CUDA: 12.8
Transformers: 5.0.0
Accelerate: 1.13.0
bitsandbytes: 0.50.2
PEFT: 0.19.1
TRL: 1.13.0
CUDA available: True
GPU: Tesla T4


NameError: name 'PY' is not defined

In [11]:
from pathlib import Path

print("Phase 5B runners found:")
runners = list(Path("/kaggle/input").rglob("scripts/run_qlora_smoke.py"))

for p in runners:
    print(" ", p)

print("\nCandidate datasets found:")
candidate_dirs = []

for p in Path("/kaggle/input").rglob("train.jsonl"):
    parent = p.parent
    if (
        (parent / "validation.jsonl").exists()
        and (parent / "policy.json").exists()
    ):
        candidate_dirs.append(parent)
        print(" ", parent)
        

Phase 5B runners found:
  /kaggle/input/datasets/hassanch6138/phase5-b-v2/scripts/run_qlora_smoke.py
  /kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/scripts/run_qlora_smoke.py

Candidate datasets found:
  /kaggle/input/datasets/hassanch6138/localsql-phase5-candidate


In [12]:
from pathlib import Path
import shutil

# -------------------------
# Discover Phase 5B source
# -------------------------
runners = [
    p for p in Path("/kaggle/input").rglob("scripts/run_qlora_smoke.py")
    if "phase5" in str(p).lower()
]

print("Runner candidates:")
for p in runners:
    print(" ", p)

assert runners, "No Phase 5B run_qlora_smoke.py found."

# Prefer the v2 source if multiple Phase 5 datasets are attached.
v2_runners = [
    p for p in runners
    if "v2" in str(p).lower()
]

runner = v2_runners[0] if len(v2_runners) == 1 else runners[0]

# runner = <repo>/scripts/run_qlora_smoke.py
SRC = runner.parent.parent

print("\nSelected Phase 5B source:")
print(SRC)

# -------------------------
# Discover candidate dataset
# -------------------------
candidate_dirs = []

for p in Path("/kaggle/input").rglob("train.jsonl"):
    parent = p.parent

    if (
        (parent / "validation.jsonl").exists()
        and (parent / "policy.json").exists()
        and "candidate" in str(parent).lower()
    ):
        candidate_dirs.append(parent)

print("\nCandidate dataset possibilities:")
for p in candidate_dirs:
    print(" ", p)

assert len(candidate_dirs) == 1, (
    f"Expected exactly 1 Phase 5 candidate dataset, found {len(candidate_dirs)}"
)

CANDIDATE = candidate_dirs[0]

# -------------------------
# Build clean working repo
# -------------------------
WORK = Path("/kaggle/working/localsql")

if WORK.exists():
    shutil.rmtree(WORK)

shutil.copytree(
    SRC,
    WORK,
    ignore=shutil.ignore_patterns(
        "localsql-phase3-src",
        ".venv",
        "__pycache__",
        ".pytest_cache",
    ),
)

candidate = WORK / "data" / "processed_phase5_candidate"
candidate.mkdir(parents=True, exist_ok=True)

for name in ["train.jsonl", "validation.jsonl", "policy.json"]:
    shutil.copy2(CANDIDATE / name, candidate / name)

print("\n--- RESULT ---")
print("Source:", SRC)
print("Candidate:", CANDIDATE)
print("Working repo:", WORK)
print("Runner:", (WORK / "scripts/run_qlora_smoke.py").exists())
print("Builder:", (WORK / "scripts/build_certification_sets.py").exists())
print("Candidate train:", (candidate / "train.jsonl").exists())
print("Validation:", (candidate / "validation.jsonl").exists())
print("Policy:", (candidate / "policy.json").exists())

Runner candidates:
  /kaggle/input/datasets/hassanch6138/phase5-b-v2/scripts/run_qlora_smoke.py
  /kaggle/input/datasets/hassanch6138/localsql-phase5-candidate/localsql-phase5a-src/scripts/run_qlora_smoke.py

Selected Phase 5B source:
/kaggle/input/datasets/hassanch6138/phase5-b-v2

Candidate dataset possibilities:
  /kaggle/input/datasets/hassanch6138/localsql-phase5-candidate

--- RESULT ---
Source: /kaggle/input/datasets/hassanch6138/phase5-b-v2
Candidate: /kaggle/input/datasets/hassanch6138/localsql-phase5-candidate
Working repo: /kaggle/working/localsql
Runner: True
Builder: True
Candidate train: True
Validation: True
Policy: True


In [13]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [14]:
!grep -n "resolve_train_examples" scripts/run_qlora_smoke.py

106:def resolve_train_examples(cfg, repo_root: Path, explicit_input: Path | None, limit: int | None):
568:        examples, input_path = resolve_train_examples(cfg, REPO_ROOT, args.input, args.max_train_examples)
579:    examples, train_path = resolve_train_examples(cfg, REPO_ROOT, args.input, args.max_train_examples)


In [16]:
!grep -n "train_path = repo_root / cfg.data.train_file" scripts/run_qlora_smoke.py

95:    train_path = repo_root / cfg.data.train_file


In [18]:
!grep -n "load_examples(cfg, REPO_ROOT" scripts/run_qlora_smoke.py

In [19]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [20]:
!python -m pip install -q -e . --no-deps

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for localsql (pyproject.toml) ... done


In [21]:
!python scripts/run_qlora_smoke.py \
    --run-id phase5-candidate-profile \
    --input data/processed_phase5_candidate/train.jsonl \
    --token-profile

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 3.73MB/s]
tokenizer_config.json: 9.38kB [00:00, 36.4MB/s]
vocab.json: 2.78MB [00:00, 21.9MB/s]
merges.txt: 1.67MB [00:00, 116MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:00<00:00, 13.2MB/s]
Tokenizer loaded (revision cdbee75f17c01a7cc42f958dc650907174af0554). Profiling 6067 examples from data/processed_phase5_candidate/train.jsonl ...
{
  "input_file": "data/processed_phase5_candidate/train.jsonl",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 6067,
  "prompt_only_token_stats": {
    "count": 6067,
    "min": 434.0,
    "median": 1796.0,
    "p90": 3328.0,
    "p95": 3705.0,
    "p99": 3849.0,
    "max": 3921.0
  },
  "prompt_only_counts_over_threshold": {
    ">3584": 316,
    ">4096": 0,
    ">8192": 0
  },
  "full_sft_sequence_token_stats": {
    "count": 6067,
    "min": 458.0,
    "median": 1842.0,
    "p90": 3383.0,
    "p95": 3735.0,
    "p99": 39

In [22]:
!python scripts/build_certification_sets.py \
    --candidate-train data/processed_phase5_candidate/train.jsonl \
    --longest-manifest data/runs/phase5-candidate-profile/train_longest_examples.json \
    --token-lengths-manifest data/runs/phase5-candidate-profile/train_token_lengths.jsonl \
    --out-dir data/certification \
    --longest-n 16 \
    --throughput-n 64

Longest-16 certification set: data/certification/longest_16.jsonl
  real length range: 3939.0 - 4005.0 tokens

Throughput sample (64 examples, REAL token lengths): data/certification/throughput_sample_64.jsonl
  {'count': 64, 'min': 458, 'median': 1841.0, 'p90': 3389.7, 'p95': 3715.2, 'p99': 3932.55, 'max': 4005, 'mean': 1935.58, 'representation_counts': {'compact_full_schema': 18, 'canonical_unchanged': 46}}


In [23]:
from pathlib import Path
import hashlib

expected = {
    "data/certification/longest_16.jsonl":
        "3629414422bb9f07b0efd2f78ef20ff7a6eb6511ca1e0bcfd1d078203a11d6bf",
    "data/certification/throughput_sample_64.jsonl":
        "321ccfbd9112106a4d91a26ee327482832e9e1d57780d8c23443110f7b99ef28",
}

for path, exp in expected.items():
    actual = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    print(path)
    print("MATCH:", actual == exp)
    print(actual)

data/certification/longest_16.jsonl
MATCH: True
3629414422bb9f07b0efd2f78ef20ff7a6eb6511ca1e0bcfd1d078203a11d6bf
data/certification/throughput_sample_64.jsonl
MATCH: True
321ccfbd9112106a4d91a26ee327482832e9e1d57780d8c23443110f7b99ef28


In [24]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
print("PYTORCH_ALLOC_CONF =", os.environ["PYTORCH_ALLOC_CONF"])


CUDA_VISIBLE_DEVICES = 0
PYTORCH_ALLOC_CONF = expandable_segments:True


In [25]:
!python scripts/run_qlora_smoke.py \
    --run-id phase5b-mem-cert \
    --input data/certification/longest_16.jsonl \
    --max-steps 2 \
    --source-revision 78b3dab4de70473b9afcab56d0156adb6daa7ab1

Loading Qwen/Qwen3-4B-Instruct-2507 for QLoRA training (4-bit nf4) ...
model.safetensors.index.json: 32.8kB [00:00, 77.6MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:57<00:00, 19.32s/it]
Download complete: 100%|████████████████████| 8.04G/8.04G [00:58<00:00, 139MB/s]
Loading weights: 100%|█| 398/398 [00:02<00:00, 142.74it/s, Materializing param=m
generation_config.json: 100%|███████████████████| 238/238 [00:00<00:00, 617kB/s]
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Trainable params: 33,030,144 / 2,238,840,320
Building completion-only-masked SFT encodings for 16 examples ...
  usable: 16  skipped (exceeds max_seq_length=4096): 0
Training (max_steps=2, resume_from_checkpoint=None) ...
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '1.382', 'grad_norm': '4.821', 'learning_rate': '0', 'epoch': '0.5'}   
{'loss': '1.212', 'grad_norm': '4.863', 'learning_rate': '0.0001', 'epoch': '1'}

In [26]:
from pathlib import Path
import json

summary_path = Path("data/runs/phase5b-mem-cert/summary.json")

print("Summary exists:", summary_path.exists())

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2))
    

Summary exists: True
{
  "run_id": "phase5b-mem-cert",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "resolved_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "quantization": {
    "load_in_4bit": true,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "trainable_param_count": 33030144,
  "total_param_count": 2238840320,
  "optimization": {
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 0.0001,
    "warmup_ratio": 0.05,
    "gradient_checkpointing": true,
    "optim": "paged_adamw_8bit",
    "seed": 42
  },
  "max_seq_length": 4096,
  "completion_only_loss": true,
  "max_steps": 2,
  "requested_train_examples"

In [27]:
config_path = Path("data/runs/phase5b-mem-cert/run_config.json")

print("Run config exists:", config_path.exists())

if config_path.exists():
    config = json.loads(config_path.read_text(encoding="utf-8"))
    print(json.dumps(config, indent=2))

Run config exists: True
{
  "run_id": "phase5b-mem-cert",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "resolved_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "quantization": {
    "load_in_4bit": true,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "task_type": "CAUSAL_LM"
  },
  "optimization": {
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 0.0001,
    "warmup_ratio": 0.05,
    "gradient_checkpointing": true,
    "optim": "paged_adamw_8bit",
    "seed": 42
  },
  "max_seq_length": 4096,
  "completion_only_loss": true,
  "max_steps": 2,
  "requested_train_examples": 16,
  "train_file": "data/certification/longest_16.jsonl",
  "train_file_sha256": "3629414422bb9f07b0efd2

In [28]:
metrics_path = Path("data/runs/phase5b-mem-cert/train_metrics.jsonl")

print("Metrics exists:", metrics_path.exists())

if metrics_path.exists():
    print(metrics_path.read_text(encoding="utf-8"))

Metrics exists: True
{"loss": 1.3817764520645142, "grad_norm": 4.821288108825684, "learning_rate": 0.0, "epoch": 0.5}
{"loss": 1.21175217628479, "grad_norm": 4.863347053527832, "learning_rate": 0.0001, "epoch": 1.0}
{"train_runtime": 524.1037, "train_samples_per_second": 0.031, "train_steps_per_second": 0.004, "total_flos": 1393080664043520.0, "train_loss": 1.296764314174652, "epoch": 1.0}



In [29]:
from pathlib import Path
import zipfile

ROOT = Path("/kaggle/working/localsql")
RUN = ROOT / "data/runs/phase5b-mem-cert"

out = Path("/kaggle/working/phase5b-mem-cert-evidence.zip")

wanted = [
    RUN / "summary.json",
    RUN / "run_config.json",
    RUN / "train_metrics.jsonl",
    RUN / "adapter_verification.json",
]

with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in wanted:
        assert p.exists(), f"Missing: {p}"
        z.write(p, arcname=p.name)

print(out)
print("exists:", out.exists())
print("size KB:", round(out.stat().st_size / 1024, 2))

/kaggle/working/phase5b-mem-cert-evidence.zip
exists: True
size KB: 2.82


In [ ]:
#GATE 3

In [34]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [35]:
!python scripts/run_qlora_smoke.py \
    --run-id phase5b-throughput \
    --input data/certification/throughput_sample_64.jsonl \
    --max-steps 8 \
    --save-steps 4 \
    --save-total-limit 2 \
    --source-revision 78b3dab4de70473b9afcab56d0156adb6daa7ab1

Loading Qwen/Qwen3-4B-Instruct-2507 for QLoRA training (4-bit nf4) ...
Loading weights: 100%|█| 398/398 [00:02<00:00, 148.71it/s, Materializing param=m
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Trainable params: 33,030,144 / 2,238,840,320
Building completion-only-masked SFT encodings for 64 examples ...
  usable: 64  skipped (exceeds max_seq_length=4096): 0
Training (max_steps=8, resume_from_checkpoint=None) ...
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '1.137', 'grad_norm': '5.71', 'learning_rate': '0', 'epoch': '0.125'}  
{'loss': '1.539', 'grad_norm': '4.791', 'learning_rate': '0.0001', 'epoch': '0.25'}
{'loss': '0.7682', 'grad_norm': '2.656', 'learning_rate': '8.571e-05', 'epoch': '0.375'}
{'loss': '0.5564', 'grad_norm': '2.199', 'learning_rate': '7.143e-05', 'epoch': '0.5'}
 50%|██████████████████████                      | 4/8 [06:59<07:15, 108.97s/it]Warning: You are sending unauthenticat

In [36]:
from pathlib import Path
import json

run = Path("data/runs/phase5b-throughput")

for name in [
    "summary.json",
    "run_config.json",
]:
    p = run / name
    print("\n" + "=" * 80)
    print(name)
    print(json.dumps(
        json.loads(p.read_text(encoding="utf-8")),
        indent=2
    ))

print("\n" + "=" * 80)
print("TRAIN METRICS")
print((run / "train_metrics.jsonl").read_text(encoding="utf-8"))


summary.json
{
  "run_id": "phase5b-throughput",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "resolved_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "quantization": {
    "load_in_4bit": true,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "trainable_param_count": 33030144,
  "total_param_count": 2238840320,
  "optimization": {
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 0.0001,
    "warmup_ratio": 0.05,
    "gradient_checkpointing": true,
    "optim": "paged_adamw_8bit",
    "seed": 42
  },
  "max_seq_length": 4096,
  "completion_only_loss": true,
  "max_steps": 8,
  "requested_train_examples": 64,

In [37]:
from pathlib import Path

checkpoint_root = Path(
    "data/runs/phase5b-throughput/checkpoint"
)

print("Checkpoint root exists:", checkpoint_root.exists())

if checkpoint_root.exists():
    for p in sorted(checkpoint_root.iterdir()):
        print(p)

Checkpoint root exists: True
data/runs/phase5b-throughput/checkpoint/checkpoint-4
data/runs/phase5b-throughput/checkpoint/checkpoint-8


In [ ]:
#GATE 4

In [38]:
from pathlib import Path

cp = Path(
    "data/runs/phase5b-throughput/checkpoint/checkpoint-4"
)

print("Exists:", cp.exists())

for p in sorted(cp.iterdir()):
    print(p.name, round(p.stat().st_size / 1024**2, 3), "MB")

Exists: True
README.md 0.005 MB
adapter_config.json 0.001 MB
adapter_model.safetensors 126.064 MB
optimizer.pt 64.558 MB
rng_state.pth 0.014 MB
scheduler.pt 0.001 MB
trainer_state.json 0.001 MB
training_args.bin 0.005 MB


In [40]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [41]:
!python scripts/run_qlora_smoke.py \
    --run-id phase5b-resume-cert \
    --input data/certification/throughput_sample_64.jsonl \
    --max-steps 6 \
    --save-steps 1 \
    --save-total-limit 2 \
    --resume-from-checkpoint data/runs/phase5b-throughput/checkpoint/checkpoint-4 \
    --source-revision 78b3dab4de70473b9afcab56d0156adb6daa7ab1

Loading Qwen/Qwen3-4B-Instruct-2507 for QLoRA training (4-bit nf4) ...
Loading weights: 100%|█| 398/398 [00:02<00:00, 148.26it/s, Materializing param=m
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Trainable params: 33,030,144 / 2,238,840,320
Building completion-only-masked SFT encodings for 64 examples ...
  usable: 64  skipped (exceeds max_seq_length=4096): 0
Training (max_steps=6, resume_from_checkpoint='data/runs/phase5b-throughput/checkpoint/checkpoint-4') ...
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
	save_steps: 1 (from args) != 4 (from trainer_state.json)
{'loss': '0.6592', 'grad_norm': '2.616', 'learning_rate': '5.714e-05', 'epoch': '0.625'}
{'loss': '0.4233', 'grad_norm': '1.385', 'learning_rate': '2e-05', 'epoch': '0.75'}
100%|█████████████████████████████████████████████| 6/6 [02:32<00:00, 29.12s/it]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable h

In [42]:
from pathlib import Path
import json

run = Path("data/runs/phase5b-resume-cert")

for name in [
    "summary.json",
    "run_config.json",
]:
    p = run / name

    print("\n" + "=" * 80)
    print(name)
    print(json.dumps(
        json.loads(p.read_text(encoding="utf-8")),
        indent=2
    ))

print("\n" + "=" * 80)
print("TRAIN METRICS")
print((run / "train_metrics.jsonl").read_text(encoding="utf-8"))


summary.json
{
  "run_id": "phase5b-resume-cert",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "resolved_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "quantization": {
    "load_in_4bit": true,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "trainable_param_count": 33030144,
  "total_param_count": 2238840320,
  "optimization": {
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 0.0001,
    "warmup_ratio": 0.05,
    "gradient_checkpointing": true,
    "optim": "paged_adamw_8bit",
    "seed": 42
  },
  "max_seq_length": 4096,
  "completion_only_loss": true,
  "max_steps": 6,
  "requested_train_examples": 64

In [43]:
checkpoint_root = Path(
    "data/runs/phase5b-resume-cert/checkpoint"
)

print("Checkpoint root exists:", checkpoint_root.exists())

if checkpoint_root.exists():
    for p in sorted(checkpoint_root.iterdir()):
        print(p)

Checkpoint root exists: True
data/runs/phase5b-resume-cert/checkpoint/checkpoint-6


In [44]:
!python scripts/run_qlora_smoke.py \
    --run-id phase5b-resume-cert-v2 \
    --input data/certification/throughput_sample_64.jsonl \
    --max-steps 8 \
    --save-steps 4 \
    --save-total-limit 2 \
    --resume-from-checkpoint data/runs/phase5b-throughput/checkpoint/checkpoint-4 \
    --source-revision 78b3dab4de70473b9afcab56d0156adb6daa7ab1

Loading Qwen/Qwen3-4B-Instruct-2507 for QLoRA training (4-bit nf4) ...
Loading weights: 100%|█| 398/398 [00:02<00:00, 144.03it/s, Materializing param=m
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Trainable params: 33,030,144 / 2,238,840,320
Building completion-only-masked SFT encodings for 64 examples ...
  usable: 64  skipped (exceeds max_seq_length=4096): 0
Training (max_steps=8, resume_from_checkpoint='data/runs/phase5b-throughput/checkpoint/checkpoint-4') ...
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '0.6592', 'grad_norm': '2.616', 'learning_rate': '5.714e-05', 'epoch': '0.625'}
{'loss': '0.4233', 'grad_norm': '1.385', 'learning_rate': '4.286e-05', 'epoch': '0.75'}
{'loss': '0.4971', 'grad_norm': '1.266', 'learning_rate': '2.857e-05', 'epoch': '0.875'}
{'loss': '0.5122', 'grad_norm': '1.518', 'learning_rate': '1.429e-05', 'epoch': '1'}
100%|█████████████████████████████████████████████| 8/8 [0

In [45]:
from pathlib import Path
import json

run = Path("data/runs/phase5b-resume-cert-v2")

for name in ["summary.json", "run_config.json"]:
    p = run / name
    print("\n" + "=" * 80)
    print(name)
    print(json.dumps(json.loads(p.read_text(encoding="utf-8")), indent=2))

print("\n" + "=" * 80)
print("TRAIN METRICS")
print((run / "train_metrics.jsonl").read_text(encoding="utf-8"))


summary.json
{
  "run_id": "phase5b-resume-cert-v2",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "resolved_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "quantization": {
    "load_in_4bit": true,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "trainable_param_count": 33030144,
  "total_param_count": 2238840320,
  "optimization": {
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 0.0001,
    "warmup_ratio": 0.05,
    "gradient_checkpointing": true,
    "optim": "paged_adamw_8bit",
    "seed": 42
  },
  "max_seq_length": 4096,
  "completion_only_loss": true,
  "max_steps": 8,
  "requested_train_examples":

In [46]:
import json
from pathlib import Path

def metric_rows(path):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        obj = json.loads(line)
        if "loss" in obj:
            rows.append(obj)
    return rows

original = metric_rows(
    "data/runs/phase5b-throughput/train_metrics.jsonl"
)

resumed = metric_rows(
    "data/runs/phase5b-resume-cert-v2/train_metrics.jsonl"
)

expected = original[4:8]

print("Original steps 5-8:", len(expected))
print("Resumed new steps:", len(resumed))

for i, (a, b) in enumerate(zip(expected, resumed), start=5):
    print(f"\nSTEP {i}")
    for field in ["loss", "grad_norm", "learning_rate"]:
        print(
            field,
            "original =", a[field],
            "resumed =", b[field],
            "match =", a[field] == b[field],
        )

Original steps 5-8: 4
Resumed new steps: 4

STEP 5
loss original = 0.6591728925704956 resumed = 0.6591728925704956 match = True
grad_norm original = 2.6162350177764893 resumed = 2.6162350177764893 match = True
learning_rate original = 5.714285714285714e-05 resumed = 5.714285714285714e-05 match = True

STEP 6
loss original = 0.4232563376426697 resumed = 0.4232563376426697 match = True
grad_norm original = 1.3854711055755615 resumed = 1.3854711055755615 match = True
learning_rate original = 4.2857142857142856e-05 resumed = 4.2857142857142856e-05 match = True

STEP 7
loss original = 0.4970863163471222 resumed = 0.4970863163471222 match = True
grad_norm original = 1.2662529945373535 resumed = 1.2662529945373535 match = True
learning_rate original = 2.857142857142857e-05 resumed = 2.857142857142857e-05 match = True

STEP 8
loss original = 0.5121994018554688 resumed = 0.5121994018554688 match = True
grad_norm original = 1.5182762145996094 resumed = 1.5182762145996094 match = True
learning_ra

In [47]:
!python scripts/export_checkpoint.py \
    --run-id phase5b-resume-cert-v2

Exported checkpoint 'checkpoint-8' from run 'phase5b-resume-cert-v2' -> /kaggle/working/localsql/data/exports/phase5b-resume-cert-v2__checkpoint-8
  global_step at export: 8
  training data packaged: data/certification/throughput_sample_64.jsonl (sha256=321ccfbd9112106a4d91a26ee327482832e9e1d57780d8c23443110f7b99ef28)
  files packaged: 14
  source revision: unavailable (origin: unavailable)

Wrote /kaggle/working/localsql/data/exports/phase5b-resume-cert-v2__checkpoint-8/MANIFEST.json and /kaggle/working/localsql/data/exports/phase5b-resume-cert-v2__checkpoint-8/RESUME_INSTRUCTIONS.md
